In [1]:
# ===== Block 1: Imports =====
import os
import random
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric
import sklearn

from torch.utils.data import Dataset

from torch_geometric.loader import DataLoader

In [2]:
# ===== Block 2: Settings =====
CACHE_PATH = "../data/graphs_cached.pt"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("====Versions:====")
print("torch:\t", torch.__version__)
print("PyG:\t", torch_geometric.__version__)
print("scikit-learn", sklearn.__version__)
print("device:\t", DEVICE)
if torch.cuda.is_available():
    print("CUDA:\t", torch.version.cuda)

====Versions:====
torch:	 2.10.0+cu128
PyG:	 2.7.0
scikit-learn 1.8.0
device:	 cuda
CUDA:	 12.8


In [3]:
class DonorAcceptorDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df: pd.DataFrame = df.reset_index(drop=True)
        #TODO: define programatically
        self.num_features = 13
        self.num_graph_features = 250
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        donor = row["D_graph"]
        acceptor = row["A_graph"]
        
        return donor, acceptor

In [4]:
if os.path.exists(CACHE_PATH):
    df = torch.load(CACHE_PATH, weights_only=False)
else:
    raise(NotImplementedError)

In [5]:
n = len(df)
idx = np.arange(n)
np.random.shuffle(idx)

cut = int(0.9 * n)
train_df = df.iloc[idx[:cut]].reset_index(drop=True)
val_df   = df.iloc[idx[cut:]].reset_index(drop=True)

train_ds = DonorAcceptorDataset(train_df)
val_ds   = DonorAcceptorDataset(val_df)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)

print("Train/Val:", len(train_ds), len(val_ds))

Train/Val: 1597 178


In [6]:
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU

from torch_geometric.nn import GATConv
from torch_geometric.utils import negative_sampling

class MolecularEncoder(nn.Module):
    def __init__(self, node_in_dim, edge_in_dim, graph_in_dim, latent_dim, hidden_dim):
        super().__init__()

        # Broadcast graph features
        in_dim = node_in_dim + graph_in_dim

        self.conv1 = GATConv(in_dim, hidden_dim, edge_dim=edge_in_dim)

        self.conv_mu = GATConv(hidden_dim, latent_dim, edge_dim=edge_in_dim)
        self.conv_logstd = GATConv(hidden_dim, latent_dim, edge_dim=edge_in_dim)

    def forward(self, x, edge_index, edge_attr, graph_attr, batch):
        
        # Broadcast graph attributes
        graph_attr_batch = graph_attr[batch]
        x = torch.cat([x, graph_attr_batch], dim=-1)

        x = F.relu(self.conv1(x, edge_index, edge_attr))

        mu = self.conv_mu(x, edge_index, edge_attr)
        logstd = self.conv_logstd(x, edge_index, edge_attr)

        return mu, logstd

class MolecularDecoder(nn.Module):
    def __init__(self, latent_dim, node_out_dim, edge_out_dim, hidden_dim):
        super().__init__()

        self.node_dec = Sequential(
            Linear(latent_dim, hidden_dim),
            ReLU(),
            Linear(hidden_dim, node_out_dim)
        )

        self.edge_prob_dec = Sequential(
            Linear(latent_dim * 2, hidden_dim),
            ReLU(),
            Linear(hidden_dim, 1)
        )

        self.edge_attr_dec = Sequential(
            Linear(latent_dim * 2, hidden_dim),
            ReLU(),
            Linear(hidden_dim, edge_out_dim)
        )

    def forward(self, z, edge_index, neg_edge_index=None):
        x_recon = self.node_dec(z)

        # edge_attr reconstruction
        z_src_pos, z_dst_pos = z[edge_index[0]], z[edge_index[1]]
        z_pos_pairs = torch.cat([z_src_pos, z_dst_pos], dim=-1)
        edge_attr_recon = self.edge_attr_dec(z_pos_pairs)

        # edge existence prediction
        if neg_edge_index is not None:
            all_edges = torch.cat([edge_index, neg_edge_index], dim=-1)
        else:
            all_edges = edge_index
        
        z_src_all, z_dst_all = z[all_edges[0]], z[all_edges[1]]
        z_all_pairs = torch.cat([z_src_all, z_dst_all], dim=-1)
        edge_logits = self.edge_prob_dec(z_all_pairs).squeeze()

        return x_recon, edge_logits, edge_attr_recon



In [7]:
class MolecularVGAE(nn.Module):
    def __init__(self, node_dim, edge_dim, graph_attr_dim, latent_dim, hidden_dim):
        super().__init__()
        self.encoder = MolecularEncoder(node_in_dim=node_dim,
                                                            edge_in_dim=edge_dim,
                                                            graph_in_dim=graph_attr_dim,
                                                            latent_dim=latent_dim,
                                                            hidden_dim=hidden_dim)

        self.decoder = MolecularDecoder(node_out_dim=node_dim,
                                                            edge_out_dim=edge_dim,
                                                            latent_dim=latent_dim,
                                                            hidden_dim=hidden_dim)

    def reparameterize(self, mu, logstd):
        if self.training:
            std = torch.exp(logstd)
            eps = torch.randn_like(std)
            return mu + eps * std
        else:
            return mu

    def encode(self, x, edge_index, edge_attr, graph_attr, batch):
        return self.encoder(x, edge_index, edge_attr, graph_attr, batch)

    def decode(self, z, edge_index, neg_edge_index=None):
        return self.decoder(z, edge_index, neg_edge_index)

    def forward(self, x, edge_index, edge_attr, graph_attr, batch, neg_edge_index=None):
        mu, logstd = self.encoder(x, edge_index, edge_attr, graph_attr, batch)
        z = self.reparameterize(mu, logstd)
        x_recon, edge_logits, edge_attr_recon = self.decoder(z, edge_index, neg_edge_index)

        return x_recon, edge_logits, edge_attr_recon, mu, logstd

    def recon_loss(self, target_data, x_recon, edge_logits, edge_attr_recon, neg_edge_index=None):
        loss_node = F.mse_loss(x_recon, target_data.x)

        loss_edge_attr = F.mse_loss(edge_attr_recon, target_data.edge_attr)

        if neg_edge_index is None:
            neg_edge_index = negative_sampling(
                edge_index=target_data.pos_edge_index,
                num_nodes=target_data.num_nodes,
                num_neg_samples=target_data.edge_index.size(1)
            )

        pos_labels = torch.ones(target_data.edge_index.size(1), device=target_data.x.device)
        neg_labels = torch.zeros(neg_edge_index.size(1), device=target_data.x.device)
        labels = torch.cat([pos_labels, neg_labels], dim=0)
        
        loss_edge_exist = F.binary_cross_entropy_with_logits(edge_logits, labels)

        return (loss_node + loss_edge_attr + loss_edge_exist).item()

    def kl_loss(self, mu, logstd):
        return -0.5 * torch.mean(torch.sum(1 + 2 * logstd - mu**2 - logstd.exp()**2, dim=1))

In [8]:
from torch_geometric.data import Data
from torch.optim import Optimizer

def train_step(model: MolecularVGAE, optimizer: Optimizer, data):
    model.train()
    optimizer.zero_grad()

    neg_edge_index = negative_sampling(
        edge_index=data.edge_index,
        num_nodes=data.num_nodes,
        num_neg_samples=data.edge_index.size(1)
    )

    x_recon, edge_logits, edge_attr_recon, mu, logstd = model(data.x,
                                                              data.edge_index,
                                                              data.edge_attr,
                                                              data.graph_attr,
                                                              data.batch,
                                                              neg_edge_index)

    recon_loss = model.recon_loss(data, x_recon, edge_logits, edge_attr_recon, neg_edge_index)

    # KL Loss
    kl_loss = model.kl_loss(mu, logstd)
    
    # TODO: kl_loss annealing? 
    loss = recon_loss + (0.001 * kl_loss)

    loss.backward()
    optimizer.step()

    return recon_loss, kl_loss


In [9]:
data = train_ds[0][0]

node_dim = data.x.size(1)
edge_dim = data.edge_attr.size(1)
graph_attr_dim = data.graph_attr.size(1)

latent_dim = 128
hidden_dim = 64

lr = 0.001

epochs = 300

model = MolecularVGAE(node_dim, edge_dim, graph_attr_dim, latent_dim, hidden_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

model = model.to(DEVICE)

for epoch in range(1, epochs+1):
    total_recon_loss = 0
    total_kl_loss = 0
    for batch in train_loader:
        # Currently processing independently, but may move to a combined D-A encoder
        donor_batch, acceptor_batch = batch
        donor_batch = donor_batch.to(DEVICE)
        acceptor_batch = acceptor_batch.to(DEVICE)
    
        d_recon_loss, d_kl_loss = train_step(model, optimizer, donor_batch)
        a_recon_loss, a_kl_loss = train_step(model, optimizer, acceptor_batch)

        total_recon_loss += (d_recon_loss+a_recon_loss)/2
        total_kl_loss += (d_kl_loss+a_kl_loss)/2

    print("Epoch: {:03d} | Recon Loss: {:.4f} | KL Loss: {:.4f}".format(epoch, total_recon_loss/train_loader.batch_size, total_kl_loss/train_loader.batch_size))


Epoch: 001 | Recon Loss: 15.5491 | KL Loss: 49.9895
Epoch: 002 | Recon Loss: 15.4835 | KL Loss: 8.1317
Epoch: 003 | Recon Loss: 15.4776 | KL Loss: 3.8571
Epoch: 004 | Recon Loss: 15.4730 | KL Loss: 2.2077
Epoch: 005 | Recon Loss: 15.4719 | KL Loss: 1.4011
Epoch: 006 | Recon Loss: 15.4704 | KL Loss: 0.9548
Epoch: 007 | Recon Loss: 15.4688 | KL Loss: 0.6823
Epoch: 008 | Recon Loss: 15.4676 | KL Loss: 0.5023
Epoch: 009 | Recon Loss: 15.4693 | KL Loss: 0.3844
Epoch: 010 | Recon Loss: 15.4658 | KL Loss: 0.2993
Epoch: 011 | Recon Loss: 15.4722 | KL Loss: 0.2393
Epoch: 012 | Recon Loss: 15.4708 | KL Loss: 0.1944
Epoch: 013 | Recon Loss: 15.4715 | KL Loss: 0.1606
Epoch: 014 | Recon Loss: 15.4669 | KL Loss: 0.1336
Epoch: 015 | Recon Loss: 15.4686 | KL Loss: 0.1129
Epoch: 016 | Recon Loss: 15.4654 | KL Loss: 0.0965
Epoch: 017 | Recon Loss: 15.4701 | KL Loss: 0.0830
Epoch: 018 | Recon Loss: 15.4703 | KL Loss: 0.0717
Epoch: 019 | Recon Loss: 15.4710 | KL Loss: 0.0625
Epoch: 020 | Recon Loss: 15.46